# Versao 13 - Pre-Processamento

Este notebook mostra como a `versao13` reaproveita a melhor base de entrada das versoes recentes e a adapta para uma arquitetura `BiGRU + MHA` com entrada univariada por timestep.


## Decisao de pre-processamento

A decisao adotada foi:

- manter a remocao das `9` features totalmente vazias da `versao11`;
- manter a reamostragem para `180` passos e a padronizacao por feature da `versao10` e `versao12`;
- gerar um novo tensor `X_seq_bigru`, com uma projecao univariada por timestep ajustada apenas no `train`, para satisfazer `F = 1` sem inflar o comprimento da sequencia para a etapa de `self-attention`.


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == 'versao13' else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import versao13.pipeline_v13 as pipeline_v13

pipeline_v13 = importlib.reload(pipeline_v13)
SELECTED_FEATURE_COLUMNS = pipeline_v13.SELECTED_FEATURE_COLUMNS
SELECTED_STATE_SENSOR_COLUMNS = pipeline_v13.SELECTED_STATE_SENSOR_COLUMNS
SELECTED_CONTINUOUS_SENSOR_COLUMNS = pipeline_v13.SELECTED_CONTINUOUS_SENSOR_COLUMNS
ALL_NULL_FEATURE_COLUMNS = pipeline_v13.ALL_NULL_FEATURE_COLUMNS
load_bundle = pipeline_v13.load_bundle
load_split_arrays = pipeline_v13.load_split_arrays
load_univariate_sequence_projection = pipeline_v13.load_univariate_sequence_projection
prepare_classification_artifacts = pipeline_v13.prepare_classification_artifacts

print('Features removidas:', ALL_NULL_FEATURE_COLUMNS)
print('Numero de features mantidas:', len(SELECTED_FEATURE_COLUMNS))
print('Numero de features continuas:', len(SELECTED_CONTINUOUS_SENSOR_COLUMNS))
print('Numero de features de estado:', len(SELECTED_STATE_SENSOR_COLUMNS))



Features removidas: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']
Numero de features mantidas: 18
Numero de features continuas: 9
Numero de features de estado: 9


In [2]:
DATASET_ROOT = PROJECT_ROOT / '3W' / 'dataset'
RUN_NAME = 'classificacao_v13_bigru_mha'

artifacts = prepare_classification_artifacts(
    dataset_root=DATASET_ROOT,
    run_name=RUN_NAME,
    random_state=42,
    sequence_length=180,
)

bundle = load_bundle(artifacts.bundle_path)
train_arrays = load_split_arrays(artifacts.split_npz_paths['train'])
validation_arrays = load_split_arrays(artifacts.split_npz_paths['validation'])
test_arrays = load_split_arrays(artifacts.split_npz_paths['test'])
projection_payload = load_univariate_sequence_projection(
    Path(artifacts.run_dir) / 'univariate_projection_v13.json'
)

print('Run dir:', artifacts.run_dir)
print('Bundle path:', artifacts.bundle_path)
print('Explained variance ratio da projecao:', round(projection_payload['explained_variance_ratio'], 6))
print('Tipo de projecao:', projection_payload['projection_type'])



Run dir: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v13/classificacao_v13_bigru_mha
Bundle path: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v13/classificacao_v13_bigru_mha/bundle_v13.json
Explained variance ratio da projecao: 0.23213
Tipo de projecao: first_principal_component_per_timestep


In [3]:
resumo_arrays = pd.DataFrame(
    [
        {
            'split': 'train',
            'X_seq': train_arrays['X_seq'].shape,
            'X_seq_bigru': train_arrays['X_seq_bigru'].shape,
            'X_tab': train_arrays['X_tab'].shape,
            'X_missing': train_arrays['X_missing'].shape,
            'X_frozen': train_arrays['X_frozen'].shape,
            'y': train_arrays['y'].shape,
        },
        {
            'split': 'validation',
            'X_seq': validation_arrays['X_seq'].shape,
            'X_seq_bigru': validation_arrays['X_seq_bigru'].shape,
            'X_tab': validation_arrays['X_tab'].shape,
            'X_missing': validation_arrays['X_missing'].shape,
            'X_frozen': validation_arrays['X_frozen'].shape,
            'y': validation_arrays['y'].shape,
        },
        {
            'split': 'test',
            'X_seq': test_arrays['X_seq'].shape,
            'X_seq_bigru': test_arrays['X_seq_bigru'].shape,
            'X_tab': test_arrays['X_tab'].shape,
            'X_missing': test_arrays['X_missing'].shape,
            'X_frozen': test_arrays['X_frozen'].shape,
            'y': test_arrays['y'].shape,
        },
    ]
)
display(resumo_arrays)



,split,X_seq,X_seq_bigru,X_tab,X_missing,X_frozen,y
0,train,"(1559, 180, 18)","(1559, 180, 1)","(1559, 162)","(1559, 180, 18)","(1559, 180, 18)","(1559,)"
1,validation,"(334, 180, 18)","(334, 180, 1)","(334, 162)","(334, 180, 18)","(334, 180, 18)","(334,)"
2,test,"(335, 180, 18)","(335, 180, 1)","(335, 162)","(335, 180, 18)","(335, 180, 18)","(335,)"


## Leitura final

A `versao13` preserva a parte mais confiavel do pipeline recente e faz apenas a adaptacao necessaria para a arquitetura `BiGRU + MHA`: transformar a serie padronizada multivariada em uma serie univariada por timestep, pronta para a atencao.
